# Costco Grocery Price Prediction: End-to-End ML Pipeline
### Senior ML Engineer Production Walkthrough

This notebook provides a complete, reproducible workflow for predicting grocery item prices from Costco catalog data:
1. **Data Exploration & Cleaning**: Parsing currencies, ratings, discounts, and handling luxury bulk outliers.
2. **Feature Engineering**: Unit extraction (oz, counts, packs), TF-IDF text representation, and rating metrics.
3. **Modeling & Cross-Validation**: 5-fold CV comparing Ridge, Random Forest, Gradient Boosting, XGBoost, and LightGBM on `log1p(Price)`.
4. **Data Leakage Check**: Comparing models trained with vs without discount attributes.
5. **Hyperparameter Tuning & Evaluation**: Bayesian optimization with Optuna and evaluation in dollar units.
6. **Artifact Export**: Saving preprocessing pipeline and trained models for the Streamlit app.

In [ ]:
import os
import sys
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.preprocessing import parse_price, parse_discount, parse_rating, build_preprocessor
from src.predict import predict_price

sns.set_theme(style='whitegrid')
print('Libraries imported successfully.')

## Step 1: Data Exploration and Cleaning
- Load raw dataset
- Inspect missing values and duplicates
- Parse Price, Discount, and Rating fields
- Handle outliers on raw and log scale

In [ ]:
df_raw = pd.read_csv('../data/costco_grocery.csv')
print(f'Raw Shape: {df_raw.shape}')
display(df_raw.head(3))

# Clean prices
df_raw['Price_clean'] = df_raw['Price'].apply(parse_price)
df_clean = df_raw[df_raw['Price_clean'].notnull() & (~df_raw['Title'].str.contains('Page Not Found', case=False, na=False))].copy()
print(f'Modeling dataset shape after removing unpriced/scraper errors: {df_clean.shape}')

# Summary stats of price
display(df_clean['Price_clean'].describe())

### EDA Visualizations
Let's display the key distributions: raw price, log price, price by subcategory, and promotional discounts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_clean['Price_clean'], bins=40, kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('Raw Price ($) - Severe Right Skew')

sns.histplot(np.log1p(df_clean['Price_clean']), bins=35, kde=True, ax=axes[1], color='#2ca02c')
axes[1].set_title('log1p(Price) - Near Normal Distribution')
plt.tight_layout()
plt.show()

## Step 2 & 4: Modeling & Data Leakage Check
### Leakage Analysis:
- **Variant A (With Discount Features)**: Feeds `discount_amount`, `discount_percent`, `has_discount` directly to the model.
- **Variant B (Without Discount Features - Recommended)**: Model learns the intrinsic base price of the product from its text, category, and pack size. The final price is computed strictly post-inference: `final_price = predicted_base_price - discount`.

Variant B avoids promotional leakage and ensures realistic inference when predicting prices of standard catalog items.

In [ ]:
# Load training metadata and model comparison results
with open('../artifacts/metadata.json', 'r') as f:
    metadata = json.load(f)

comp_df = pd.DataFrame(metadata['model_comparisons'])
print('=== MODEL COMPARISON TABLE ON TEST SET ===')
display(comp_df)

## Step 3 & 5: Tuned Model Evaluation & Top Feature Drivers
Displaying the tuned Random Forest performance and top Gini importance drivers.

In [ ]:
print('Tuned Model Performance on Hold-out Test Set:')
for k, v in metadata['metrics'].items():
    print(f'  {k}: {v}')

top_feats = pd.DataFrame(metadata['top_features'])
plt.figure(figsize=(10, 6))
plt.barh(top_feats.head(12)['feature'].iloc[::-1], top_feats.head(12)['importance'].iloc[::-1], color='#1f77b4')
plt.title('Top Feature Drivers of Costco Grocery Price')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Step 6: Live Prediction Test
Simulating live user query through the production prediction module.

In [ ]:
sample_prediction = predict_price(
    title='Kirkland Signature Organic Extra Virgin Olive Oil, 2 L',
    sub_category='Pantry & Dry Goods',
    feature='Cold extracted, 100% Italian olives',
    product_description='Rich taste, kosher, perfect for cooking and salads',
    rating=4.8,
    discount_val=3.0,
    discount_type='amount',
    model_path='../artifacts/model.joblib',
    metadata_path='../artifacts/metadata.json'
)

print(json.dumps(sample_prediction, indent=2))